# 02 - Feature Engineering & Preprocessing

## Objectives

The objective of this notebook is to address the data quality issues identified in EDA 
and prepare the dataset for modeling. This includes binarizing the target variable from 
0-4 to 0-1, mode imputation for missing values in ca and thal, grouping rare categorical 
categories, log transforming oldpeak, handling outliers, and scaling continuous features.

Given the small sample size of 303 patients, preserving as much data as possible is a 
priority — this motivated the choice of mode imputation over dropping rows for missing 
values. All transformations are applied after the train/test split to prevent data leakage, 
meaning imputation and scaling parameters are learned from training data only and then 
applied to the test set.

## Output
- The output of this notebook is a cleaned, transformed, and scaled dataset saved to `data/processed/heart_explored.csv` and ready for modeling in notebook 03.

## 2.1 Setup & Load Data

Importing the same core libraries as notebook 01 with the addition of sklearn modules for train/test splitting, imputation, and scaling. Path constants and random state are defined here to ensure reproducibility and consistent file references throughout the notebook.

In [3]:
import os
import warnings
warnings.filterwarnings('ignore') # suppress sklearn warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

sns.set_theme()
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_STATE = 42
RAW_DATA_PATH = '../data/raw/heart.csv'
PROCESSED_DATA_PATH = '../data/processed/heart_cleaned.csv'

Loading the raw dataset fresh from data/raw/ to ensure all transformations in this notebook start from the original unmodified data. Column names applied manually as in notebook 01.

In [4]:
columns = ['age','sex','cp', 'trestbps','chol', 'fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','target']
df = pd.read_csv(RAW_DATA_PATH, names = columns, header = None)
print(df.shape)
df.head()

(303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 2.2 Target Binarization

Binarizing the target variable from its raw 0-4 encoding to binary 0/1. The values 1-4 all indicate presence of heart disease at varying severity - for this binary classification I only need to distinguish disease present vs not present. This decision was motivated by the consistent overlap between categories 1-4 observed in bivariate analysis. 

In [5]:
df['target'] = (df['target']>0).astype(int)

In [4]:
df['target'].value_counts()

target
0    164
1    139
Name: count, dtype: int64

Target was successfully binarized. 164 patients with no disease (0) and 139 with disease present (1), confirming the ~54/46 split identified in EDA. 

## 2.3 Categorical Feature Engineering

In EDA I identified that restecg, thal, slope, and ca have rare categories that could cause instability during modeling. Grouping these reduces noise by ensuring the model has enough examples to learn reliable patterns from each category.

For restecg, category 1 is extremely rare (~1% of patients). Since both 1 and 2 represent abnormal ECG results, I combine them into a single abnormal category (1) and leave 0 as normal.

For thal, category 6 (fixed defect) is very rare. Although fixed and reversible defects are clinically distinct, the sample size for category 6 is too small to be reliable, so I group it with category 7 (reversible defect).

For slope, category 3 is rare. Since category 2 (flat) had the most disease cases in bivariate analysis and is the most informative, I group upsloping (1) and downsloping (3) together and leave flat (2) as is.

For ca, categories 1-3 all represent one or more blocked vessels. Given the rarity of ca=3, I combine 1, 2, and 3 into a single category (1) representing any blocked vessels, and leave 0 as no blocked vessels. This also creates a clinically meaningful binary question: are any vessels blocked?

In [6]:
df['restecg'] = df['restecg'].map({0.0: 0.0, 1.0: 1.0, 2.0: 1.0})
df['thal'] = df['thal'].map({'3.0': 3.0, '6.0': 7.0, '7.0': 7.0, '?': np.nan})
df['slope'] = df['slope'].map({1.0: 1.0, 2.0: 2.0, 3.0: 1.0})
df['ca'] = df['ca'].map({'0.0': 0.0, '1.0': 1.0, '2.0': 1.0, '3.0': 1.0, '?': np.nan})

In [7]:
print(df['restecg'].value_counts())
print(df['thal'].value_counts())
print(df['slope'].value_counts())
print(df['ca'].value_counts())

restecg
1.0    152
0.0    151
Name: count, dtype: int64
thal
3.0    166
7.0    135
Name: count, dtype: int64
slope
1.0    163
2.0    140
Name: count, dtype: int64
ca
0.0    176
1.0    123
Name: count, dtype: int64


Category groupings confirmed. Missing values in ca (4) and thal (2) preserved as NaN for imputation in section 2.5. Restecg and slope show no missing values as expected.

## 2.4 Train/Test Split

Before imputing and scaling, I split the data into training and test sets to avoid data leakage. The split follows a standard 80/20 ratio — 242 patients for training and 61 for testing. By splitting first, all transformation parameters such as imputation values and 
scaling statistics are learned from the training set only and then applied to both sets. This ensures the test set remains a true representation of unseen data and the model is evaluated fairly.

In [10]:
X = df.drop('target', axis = 1)
y = df['target']
X_train, X_split, y_train, y_split = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_STATE)

In [11]:
print(X_train.shape)
print(X_split.shape)
print(y_train.shape)
print(y_split.shape)

(242, 13)
(61, 13)
(242,)
(61,)


## 2.5 Missing Value Imputation

## 2.6 Outlier Handling

## 2.7 Log Transform

## 2.8 Feature Scaling

## 2.9 Save Processed Data